### Cell 01 - Import Dependencies and Basic Settings

This cell imports libraries for numerical computing, plotting, table I/O, and GP-HT-related routines, and sets basic paths or plotting styles used later.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.


In [ ]:
from __future__ import annotations
import math
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from math import pi
from IPython.display import Image, display
# GP_hilbert.py is imported from the current folder. It is based on the companion code of the original GP-HT work by Ciucci et al. (J. Electrochem. Soc. 2020, DOI: 10.1149/1945-7111/aba9c0).
import GP_hilbert as gpf
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Microsoft YaHei", "SimHei"],
        "axes.unicode_minus": False,
        "font.size": 36,
        "axes.labelsize": 48,
        "xtick.labelsize": 36,
        "ytick.labelsize": 36,
        "legend.fontsize": 24,
        "axes.linewidth": 3,
        "legend.frameon": False,
    }
)


### Cell 02 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.
- Main parameters: INPUT_EXCEL, OUTPUT_ROOT, RUN_ID, DATA_DIR, FIG_DIR, SHEET_NAME_PATTERNS, MAX_SHEETS, SKIP_INVALID_SHEETS, SORT_BY_FREQUENCY, FILTER_POSITIVE_IMAG, INCLUDE_RAW_ANALYSIS, NOISE_LEVELS, SPARSE_RATIOS, LIMITED_PERCENT_LIST, LIMITED_RANGE_LIST, LIMITED_RANGE_MIN_POINTS, LIMITED_RANGE_FALLBACK_HALF_WINDOW, RANDOM_SEED.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.


In [ ]:
# User configuration
# Input file: update this path before running with your own data.
NOTEBOOK_DIR = Path.cwd()
CASE_DATA_DIR = NOTEBOOK_DIR.parent / "6 Case Data" if (NOTEBOOK_DIR.parent / "6 Case Data").exists() else NOTEBOOK_DIR / "6 Case Data"  # This is the author's local input/output path; please update it before running.
DATASET_NAME = "cell"  # Options: "cell", "potato", "mouse"。
CASE_DATA_FILES = {
    "cell": CASE_DATA_DIR / "case_data_cell.xlsx",
    "potato": CASE_DATA_DIR / "case_data_potato.xlsx",
    "mouse": CASE_DATA_DIR / "case_data_mouse.xlsx",
}
INPUT_EXCEL = CASE_DATA_FILES[DATASET_NAME]  # This is the author's local input/output path; please update it before running.
# Output directory: each run is saved under data/RUN_ID and figures/RUN_ID to avoid overwriting previous results.
OUTPUT_ROOT = Path(r"C:/Users/CYJ/Desktop/DRT-GPHT_experimental_data_outputs")  # This is the author's local input/output path; please update it before running.
RUN_ID = datetime.now().strftime("%y%m%d%H%M")
DATA_DIR = OUTPUT_ROOT / "data" / RUN_ID  # This is the author's local input/output path; please update it before running.
FIG_DIR = OUTPUT_ROOT / "figures" / RUN_ID  # This is the author's local input/output path; please update it before running.
# Sheet selection; an empty list reads all sheets.
SHEET_NAME_PATTERNS: list[str] = []
MAX_SHEETS: int | None = None
SKIP_INVALID_SHEETS = True
# Data preprocessing.
SORT_BY_FREQUENCY = True  # True means sorting by ascending frequency after reading, which facilitates later interpolation, truncation, and plotting.
FILTER_POSITIVE_IMAG = False  # In bioimpedance data, imag>0 usually suggests measurement error, contact problems, or high-frequency inductive artifacts; points are retained by default and can be filtered during quality control.
# Degraded-condition settings: multiple noise, sparsity, or frequency-limitation settings can be run at once.
INCLUDE_RAW_ANALYSIS = True
NOISE_LEVELS = [0.10]  # Multiplicative relative noise levels to generate; 0.10 means Z_new = Z * (1 + N(0, 0.10)).
SPARSE_RATIOS = [3]  # Sparse-sampling ratio; 3 means keeping one point every three frequency points.
# Point-count truncation: 0.20 removes 10% of points from each low- and high-frequency end.
LIMITED_PERCENT_LIST: list[float] = [0.20]  # Point-count truncation settings; 0.20 removes 10% of points from each low- and high-frequency end.
# Frequency-range truncation: each tuple gives the retained frequency interval, e.g., (3000.0, 1000000.0).
LIMITED_RANGE_LIST: list[tuple[float, float]] = []  # Frequency-range retention settings; each tuple is (minimum frequency, maximum frequency), and an empty list disables this condition.
LIMITED_RANGE_MIN_POINTS = 5  # Minimum number of points retained after frequency-range truncation; a fallback window is used below this value.
LIMITED_RANGE_FALLBACK_HALF_WINDOW = 10  # Half-window size retained around the middle frequency point when range truncation leaves too few points.
# Noise and prediction settings.
RANDOM_SEED = 42
PRED_FREQ_MIN = 1.0e1
PRED_FREQ_MAX_DEFAULT = 1.0e7
RAW_PRED_MAX_USES_OBS_MAX = True
N_PRED = 200
# Plotting settings.
FIG_DPI = 100  # Resolution used when saving PNG figures.
RAW_LINE_WIDTH = 3.0  # Line width of the raw reference curve.
GP_LINE_WIDTH = 4.0  # Line width of the GP-HT prediction curve.
INPUT_MARKER_SIZE = 8.0  # Marker size for degraded input points.
CI_ALPHA_1SIGMA = 0.90  # Transparency of the one-standard-deviation credible-interval band.
CI_ALPHA_2SIGMA = 0.65  # Transparency of the two-standard-deviation credible-interval band.
CI_ALPHA_3SIGMA = 0.35  # Transparency of the three-standard-deviation credible-interval band.


### Cell 03 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.

- Function descriptions: 
  - `ensure_dir(path)`: Takes a directory path, creates missing directories, and returns the same `Path` object for later table or figure output.
  - `safe_name(text, max_len)`: Takes a sample or condition name, removes invalid filename characters, limits its length, and returns a safe filename stem.
  - `safe_sheet_name(base, used)`: Takes a candidate sheet name and the set of used names, and returns a valid, unique Excel sheet name.
  - `token_float(value)`: Takes a noise level or ratio, replaces the decimal point with `p`, and returns a token suitable for file or condition names.
  - `make_conditions()`: Generates the list of raw, noisy, sparse, and limited conditions from the global configuration.
  - `select_sheets(xls_file)`: Reads sheet names from the Excel workbook and filters samples by user-defined keywords and maximum count.
  - `standardize_input_sheet(df)`: Takes one raw sheet, reads frequency, real, and imaginary columns, removes invalid values, sorts the data, performs basic quality control, and returns a standardized impedance dictionary.
  - `apply_condition(data_raw, condition)`: Takes the raw impedance data and one degraded condition, and returns the corresponding raw/noisy/sparse/limited input spectrum.


In [ ]:
@dataclass(frozen=True)
class Condition:
    """Configuration container for one degraded input condition.
    
    Fields:
        name: label used in output columns, Excel sheets, logs, and figure legends.
        kind: condition type, including raw, noisy, sparse, limited_pct, limited_range, and limited_percent.
        value: condition parameter, such as noise level, sparse ratio, retained fraction, or frequency range.
    Purpose:
        Keep degraded-condition settings explicit and reproducible across the workflow.
    """
    name: str
    kind: str
    value: float | int | tuple[float, float] | None = None
def ensure_dir(path: Path) -> Path:
    """Create the directory and return its path."""
    path.mkdir(parents=True, exist_ok=True)
    return path
def safe_name(text: object, max_len: int = 80) -> str:
    """Remove characters that are invalid in Windows file names."""
    value = str(text).strip()
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"\s+", "_", value)
    return value[:max_len] if value else "sample"
def safe_sheet_name(base: str, used: set[str]) -> str:
    """Generate a valid, unique Excel sheet name with at most 31 characters."""
    cleaned = re.sub(r"[\[\]:*?/\\]", "_", str(base))[:31] or "sheet"
    name = cleaned
    index = 1
    while name in used:
        suffix = f"_{index}"
        name = cleaned[: 31 - len(suffix)] + suffix
        index += 1
    used.add(name)
    return name
def token_float(value: float) -> str:
    """Convert a decimal value into a token suitable for condition names, e.g., 0.1 -> 0p1."""
    return f"{value:g}".replace(".", "p")
def make_conditions() -> list[Condition]:
    """Generate the list of degraded conditions requested by the user configuration.
    
    Inputs:
        Uses INCLUDE_RAW_ANALYSIS, NOISE_LEVELS, SPARSE_RATIOS, LIMITED_PERCENT_LIST, and LIMITED_RANGE_LIST.
    Outputs:
        List of Condition objects.
    Purpose:
        Build raw, noisy, sparse, and limited inputs in a unified way before batch processing.
    """
    conditions: list[Condition] = []
    if INCLUDE_RAW_ANALYSIS:
        conditions.append(Condition("raw", "raw"))
    for ratio in SPARSE_RATIOS:
        conditions.append(Condition(f"sparse_{int(ratio)}", "sparse", int(ratio)))
    for low, high in LIMITED_RANGE_LIST:
        # Use the concise name limited when there is only one range-limited condition.
        name = "limited" if len(LIMITED_RANGE_LIST) == 1 else f"limited_range_{low:g}_{high:g}"
        conditions.append(Condition(name, "limited_range", (float(low), float(high))))
    for pct in LIMITED_PERCENT_LIST:
        conditions.append(Condition(f"limited_{int(round(float(pct) * 100))}pct", "limited_percent", float(pct)))
    for level in NOISE_LEVELS:
        conditions.append(Condition(f"noisy_{token_float(float(level))}", "noisy", float(level)))
    return conditions
def select_sheets(xls_file: pd.ExcelFile) -> list[str]:
    """Select worksheet names according to optional name filters and maximum-count settings.
    
    Inputs:
        xls_file or xls: an opened Excel workbook.
    Outputs:
        List of sheet names to analyze.
    Purpose:
        Read all sample sheets by default, or restrict the run for debugging or targeted analysis.
    """
    sheets = list(xls_file.sheet_names)
    if SHEET_NAME_PATTERNS:
        keys = [key.lower() for key in SHEET_NAME_PATTERNS]
        sheets = [sheet for sheet in sheets if any(key in sheet.lower() for key in keys)]
    if MAX_SHEETS is not None:
        sheets = sheets[: int(MAX_SHEETS)]
    return sheets
def standardize_input_sheet(df: pd.DataFrame) -> dict[str, np.ndarray]:
    """Read the first three columns of one sheet and return an impedance-data dictionary.
    
    Inputs:
        df: DataFrame read from one Excel sheet; the first three columns are interpreted as frequency, Re(Z), and Im(Z).
    Outputs:
        Dictionary containing freq, re, and im NumPy arrays.
    Purpose:
        Remove invalid values, optionally filter positive Im(Z), and sort the spectrum by ascending frequency.
    """
    if df.shape[1] < 3:
        raise ValueError("The input sheet must contain at least three columns: freq, re, and im.")
    freq = df.iloc[:, 0].to_numpy(dtype=float)
    z_re = df.iloc[:, 1].to_numpy(dtype=float)
    z_im = df.iloc[:, 2].to_numpy(dtype=float)
    valid = np.isfinite(freq) & np.isfinite(z_re) & np.isfinite(z_im) & (freq > 0)
    if FILTER_POSITIVE_IMAG:
        valid &= z_im <= 0
    freq = freq[valid]
    z_re = z_re[valid]
    z_im = z_im[valid]
    if SORT_BY_FREQUENCY:
        order = np.argsort(freq)
        freq = freq[order]
        z_re = z_re[order]
        z_im = z_im[order]
    return {"freq": freq, "re": z_re, "im": z_im}
def apply_condition(data_raw: dict[str, np.ndarray], condition: Condition) -> dict[str, np.ndarray]:
    """Apply one degraded condition to the raw impedance spectrum.
    
    Inputs:
        raw or data_raw: raw impedance data for one sample.
        condition: Condition object generated by make_conditions.
        sample_name: sample/sheet name used to create reproducible noisy inputs when applicable.
    Outputs:
        DataFrame or dictionary with the same impedance fields as the input.
    Purpose:
        Copy raw data, add multiplicative relative noise, perform sparse sampling, or keep the requested frequency subset.
    """
    freq = data_raw["freq"]
    z_re = data_raw["re"]
    z_im = data_raw["im"]
    if condition.kind == "raw":
        return {"freq": freq.copy(), "re": z_re.copy(), "im": z_im.copy()}
    if condition.kind == "sparse":
        ratio = max(int(condition.value), 1)
        indices = np.arange(0, len(freq), ratio)
        return {"freq": freq[indices], "re": z_re[indices], "im": z_im[indices]}
    if condition.kind == "limited_range":
        low, high = condition.value
        mask = (freq >= low) & (freq <= high)
        if np.sum(mask) < LIMITED_RANGE_MIN_POINTS:
            mid = len(freq) // 2
            start = max(0, mid - LIMITED_RANGE_FALLBACK_HALF_WINDOW)
            end = min(len(freq), mid + LIMITED_RANGE_FALLBACK_HALF_WINDOW)
            mask = np.zeros_like(freq, dtype=bool)
            mask[start:end] = True
        return {"freq": freq[mask], "re": z_re[mask], "im": z_im[mask]}
    if condition.kind == "limited_percent":
        pct = min(max(float(condition.value), 0.0), 0.95)
        n = len(freq)
        drop_each_side = int(math.floor(n * pct / 2))
        if drop_each_side > 0 and n - 2 * drop_each_side >= LIMITED_RANGE_MIN_POINTS:
            indexer = slice(drop_each_side, n - drop_each_side)
        else:
            indexer = slice(None)
        return {"freq": freq[indexer], "re": z_re[indexer], "im": z_im[indexer]}
    if condition.kind == "noisy":
        # Multiplicative relative noise: Z_new = Z * (1 + N(0, sigma)).
        level = float(condition.value)
        noise_re = np.random.normal(0, level, len(z_re))
        noise_im = np.random.normal(0, level, len(z_im))
        return {"freq": freq.copy(), "re": z_re * (1 + noise_re), "im": z_im * (1 + noise_im)}
    raise ValueError(f"Unknown degraded condition: {condition.kind}")


### Cell 04 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.

- Function descriptions: 
  - `prepare_datasets()`: Reads multiple sample sheets from the input Excel workbook and constructs impedance-data dictionaries and the condition list for all degraded conditions.


In [ ]:
def prepare_datasets() -> tuple[dict[str, dict[str, dict[str, np.ndarray]]], list[Condition]]:
    """Read the Excel workbook and generate all degraded datasets.
    
    Inputs:
        Uses INPUT_EXCEL and the global condition/preprocessing configuration.
    Outputs:
        datasets_pool: nested dictionary indexed by sample name and condition name.
        conditions: list of Condition objects used in this run.
    Purpose:
        Centralize workbook reading, preprocessing, degraded-input generation, and reproducible random seeding.
    """
    ensure_dir(DATA_DIR)
    ensure_dir(FIG_DIR)
    np.random.seed(RANDOM_SEED)
    xls_file = pd.ExcelFile(INPUT_EXCEL)
    sheets = select_sheets(xls_file)
    conditions = make_conditions()
    datasets_pool: dict[str, dict[str, dict[str, np.ndarray]]] = {}
    print(f"Workbook loaded successfully. Number of sheets: {len(xls_file.sheet_names)}")
    print("Sheets selected for this run:", sheets)
    print("Conditions selected for this run:", [condition.name for condition in conditions])
    for sheet in sheets:
        try:
            raw = standardize_input_sheet(pd.read_excel(xls_file, sheet_name=sheet))
        except Exception as exc:
            print(f"Skipped sheet: {sheet} | {exc}")
            if SKIP_INVALID_SHEETS:
                continue
            raise
        datasets_pool[sheet] = {}
        for condition in conditions:
            datasets_pool[sheet][condition.name] = apply_condition(raw, condition)
    print("Data preprocessing completed.")
    return datasets_pool, conditions
datasets_pool, conditions = prepare_datasets()


### Cell 05 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.

- Function descriptions: 
  - `gpht_im_input_drt(data, condition_name)`: Takes one impedance spectrum, uses the imaginary component as the GP-HT input, regresses the imaginary component, predicts the real component through the Hilbert cross-kernel, and returns predicted curves and credible intervals.
  - `result_to_dataframe(condition_name, result)`: Converts one GP-HT result into a table containing prediction frequency, real component, imaginary component, and standard deviations.
  - `pad_dataframe(df, length)`: Pads tables of different lengths to a common length so they can be concatenated side by side in one Excel sheet.


In [ ]:
def gpht_im_input_drt(data: dict[str, np.ndarray], condition_name: str) -> dict:
    """Run the imInput DRT-GPHT calculation for one impedance spectrum.
    
    Inputs:
        data: dictionary with freq, re, and im arrays for the current condition.
        condition_name: condition label used for frequency-range handling and logging.
    Outputs:
        Dictionary containing experimental input data, GP-HT predictions, optimized theta values, and kernel options.
    Purpose:
        Use Im(Z) as the GP-HT input, predict Re(Z) through the Hilbert cross-covariance, regress Im(Z), and compute posterior standard deviations.
    """
    freq_vec = np.asarray(data["freq"], dtype=float)
    z_re_exp = np.asarray(data["re"], dtype=float)
    z_im_exp = np.asarray(data["im"], dtype=float)
    omega_vec = 2.0 * pi * freq_vec
    n_freqs = len(freq_vec)
    # Initial DRT-GPHT parameters; these values can be used as starting points for later sensitivity analysis.
    min_freq = 1e-1
    current_min = min(np.min(freq_vec), min_freq)
    tau_max = 10 ** (np.ceil(np.log10(1 / (2 * pi * current_min))) + 1)
    min_noise = np.mean(np.abs(z_im_exp)) * 0.0001
    ker_opts = {
        "sigma_DRT": np.std(z_im_exp) * 0.1,
        "sigma_SB": 0.1,
        "ell": 1.0,
        "tau_max": tau_max,
        "DRT": True,
        "SB": True,
        "SB_ker_type": "IQ",
    }
    theta_0 = np.array([min_noise * 5, np.std(z_im_exp) * 0.5, 1.0, 1.0, 1e-9])
    theta_opt, ker_opts_opt = gpf.compute_opt_theta(theta_0, ker_opts, freq_vec, z_im_exp, type_data="im")
    sigma_n, sigma_DRT, sigma_SB, ell, sigma_L = theta_opt
    # Build the full input-side covariance and use Cholesky decomposition to obtain inv_K.
    K_im = gpf.mat_K(omega_vec, omega_vec, ker_opts_opt, "im")
    Sigma = (sigma_n**2) * np.eye(n_freqs)
    K_full = K_im + Sigma + (sigma_L**2) * np.outer(omega_vec, omega_vec)
    if not gpf.is_PD(K_full):
        K_full = gpf.nearest_PD(K_full)
    L_mat = np.linalg.cholesky(K_full)
    inv_L = np.linalg.inv(L_mat)
    inv_K = inv_L.T @ inv_L
    pred_max = np.max(freq_vec) if (condition_name == "raw" and RAW_PRED_MAX_USES_OBS_MAX) else PRED_FREQ_MAX_DEFAULT
    freq_pred = np.logspace(np.log10(PRED_FREQ_MIN), np.log10(pred_max), N_PRED)
    omega_pred = 2.0 * pi * freq_pred
    mu_re_pred = np.zeros_like(freq_pred)
    mu_im_pred = np.zeros_like(freq_pred)
    sigma_re_pred = np.zeros_like(freq_pred)
    sigma_im_pred = np.zeros_like(freq_pred)
    # Compute posterior mean and standard deviation point by point at each prediction frequency.
    for i, w in enumerate(omega_pred):
        w_np = np.array([w])
        # cross covariance: byimaginary componentinputpredictionreal component.
        k_re = gpf.mat_K(omega_vec, w_np, ker_opts_opt, "im-re").flatten()
        k_re_re = gpf.mat_K(w_np, w_np, ker_opts_opt, "re").flatten()
        # Same-side covariance: posterior regression for the input imaginary component.
        k_im = gpf.mat_K(omega_vec, w_np, ker_opts_opt, "im").flatten() + (sigma_L**2) * omega_vec * w_np
        k_im_im = gpf.mat_K(w_np, w_np, ker_opts_opt, "im").flatten() + (sigma_L**2) * w_np**2
        mu_re_pred[i] = k_re @ (inv_K @ z_im_exp)
        var_re = sigma_n**2 + k_re_re - k_re @ (inv_K @ k_re)
        sigma_re_pred[i] = np.sqrt(np.abs(var_re))
        mu_im_pred[i] = k_im @ (inv_K @ z_im_exp)
        var_im = sigma_n**2 + k_im_im - k_im @ (inv_K @ k_im)
        sigma_im_pred[i] = np.sqrt(np.abs(var_im))
    # R_inf alignment: the GP-HT cross-predicted Re(Z) may not contain the constant real-component baseline.
    # Compare Z_re_exp with the predicted real component on the measured frequencies,
    # then use the mean residual as the global offset.
    interp_re = np.interp(freq_vec, freq_pred, mu_re_pred)
    R_inf = np.mean(z_re_exp - interp_re)
    z_re_final = mu_re_pred + R_inf
    return {
        "exp": {"f": freq_vec, "re": z_re_exp, "im": z_im_exp},
        "gp": {"f": freq_pred, "re": z_re_final, "im": mu_im_pred, "sigma_re": sigma_re_pred, "sigma_im": sigma_im_pred},
        "theta": {
            "sigma_n": sigma_n,
            "sigma_DRT": sigma_DRT,
            "sigma_SB": sigma_SB,
            "ell": ell,
            "sigma_L": sigma_L,
            "tau_max": tau_max,
            "R_inf": R_inf,
        },
        "ker_opts": ker_opts_opt,
    }
def result_to_dataframe(condition_name: str, result: dict) -> pd.DataFrame:
    """Convert one DRT-GPHT result into an exportable table.
    
    Inputs:
        condition_name: label used as the output-column prefix.
        result: dictionary returned by gpht_im_input_drt.
    Outputs:
        DataFrame containing experimental input data, GP-HT predictions, and 1/2/3 sigma credible-interval columns.
    Purpose:
        Put measured and reconstructed curves on a common table structure for Excel export.
    """
    exp = result["exp"]
    gp = result["gp"]
    df_exp = pd.DataFrame(
        {
            f"{condition_name}_Freq": exp["f"],
            f"{condition_name}_Re_Exp": exp["re"],
            f"{condition_name}_Im_Exp": exp["im"],
            f"{condition_name}_NegIm_Exp": -exp["im"],
        }
    )
    df_gp = pd.DataFrame(
        {
            f"{condition_name}_Freq_GP": gp["f"],
            f"{condition_name}_Re_GP": gp["re"],
            f"{condition_name}_Im_GP": gp["im"],
            f"{condition_name}_NegIm_GP": -gp["im"],
            f"{condition_name}_SigmaRe_GP": gp["sigma_re"],
            f"{condition_name}_SigmaIm_GP": gp["sigma_im"],
        }
    )
    for level in (1, 2, 3):
        df_gp[f"{condition_name}_Re_GP_Lower_{level}sigma"] = gp["re"] - level * gp["sigma_re"]
        df_gp[f"{condition_name}_Re_GP_Upper_{level}sigma"] = gp["re"] + level * gp["sigma_re"]
        df_gp[f"{condition_name}_Im_GP_Lower_{level}sigma"] = gp["im"] - level * gp["sigma_im"]
        df_gp[f"{condition_name}_Im_GP_Upper_{level}sigma"] = gp["im"] + level * gp["sigma_im"]
    max_len = max(len(df_exp), len(df_gp))
    return pd.concat([pad_dataframe(df_exp, max_len), pad_dataframe(df_gp, max_len)], axis=1)
def pad_dataframe(df: pd.DataFrame, length: int) -> pd.DataFrame:
    """Pad a DataFrame to a specified length for side-by-side export.
    
    Inputs:
        df: table to pad.
        length: target row count.
    Outputs:
        DataFrame with the requested row count.
    Purpose:
        Allow raw, degraded, and predicted data with different lengths to be written side by side.
    """
    if len(df) >= length:
        return df.reset_index(drop=True)
    padding = pd.DataFrame(index=range(length - len(df)), columns=df.columns)
    return pd.concat([df.reset_index(drop=True), padding], ignore_index=True)


### Cell 06 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.

- Function descriptions: 
  - `build_run_info()`: Collects run information such as input path, output path, degraded conditions, and prediction frequencies, and returns a log table.
  - `run_gpht_and_save()`: Runs DRT-GPHT in batch and writes raw data, degraded inputs, and prediction results for each sample and condition to Excel.


In [ ]:
def build_run_info() -> pd.DataFrame:
    """generatethistable, writeworkbook sheet."""
    return pd.DataFrame(
        {
            "parameter": [
                "INPUT_EXCEL",
                "RUN_ID",
                "DATA_DIR",
                "FIG_DIR",
                "SHEET_NAME_PATTERNS",
                "SKIP_INVALID_SHEETS",
                "SORT_BY_FREQUENCY",
                "FILTER_POSITIVE_IMAG",
                "NOISE_LEVELS",
                "SPARSE_RATIOS",
                "LIMITED_PERCENT_LIST",
                "LIMITED_RANGE_LIST",
                "PRED_FREQ_MIN",
                "PRED_FREQ_MAX_DEFAULT",
                "RAW_PRED_MAX_USES_OBS_MAX",
                "N_PRED",
                "RANDOM_SEED",
            ],
            "value": [
                str(INPUT_EXCEL),
                RUN_ID,
                str(DATA_DIR),
                str(FIG_DIR),
                ", ".join(SHEET_NAME_PATTERNS) if SHEET_NAME_PATTERNS else "ALL",
                SKIP_INVALID_SHEETS,
                SORT_BY_FREQUENCY,
                FILTER_POSITIVE_IMAG,
                str(NOISE_LEVELS),
                str(SPARSE_RATIOS),
                str(LIMITED_PERCENT_LIST),
                str(LIMITED_RANGE_LIST),
                PRED_FREQ_MIN,
                PRED_FREQ_MAX_DEFAULT,
                RAW_PRED_MAX_USES_OBS_MAX,
                N_PRED,
                RANDOM_SEED,
            ],
        }
    )
def run_gpht_and_save() -> tuple[dict, Path, pd.DataFrame]:
    """Run imInput DRT-GPHT in batch and save the Excel workbook.
    
    Inputs:
        Uses datasets_pool and conditions generated by prepare_datasets.
    Outputs:
        results_store: nested dictionary of all sample/condition results.
        excel_path: path of the exported workbook.
        log_df: optimization and status log for each sample and condition.
    Purpose:
        Process all selected samples and degraded conditions, then export data tables and logs.
    """
    ensure_dir(DATA_DIR)
    results_store: dict[str, dict[str, dict]] = {}
    log_rows: list[dict] = []
    excel_path = DATA_DIR / "GPHT_experimental_results_with_credible_intervals.xlsx"
    used_sheet_names: set[str] = set()
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        build_run_info().to_excel(writer, sheet_name="run_info", index=False)
        for sheet, group in datasets_pool.items():
            print(f"\nProcessing sample: {sheet}")
            results_store[sheet] = {}
            frames: list[pd.DataFrame] = []
            for condition in conditions:
                data = group[condition.name]
                try:
                    result = gpht_im_input_drt(data, condition.name)
                    results_store[sheet][condition.name] = result
                    frames.append(result_to_dataframe(condition.name, result))
                    log_rows.append({"sheet": sheet, "condition": condition.name, "success": True, **result["theta"]})
                    print(f"  Completed: {condition.name}")
                except Exception as exc:
                    log_rows.append({"sheet": sheet, "condition": condition.name, "success": False, "message": repr(exc)})
                    print(f"  Failed: {condition.name} | {exc}")
            if frames:
                max_len = max(len(frame) for frame in frames)
                merged = pd.concat([pad_dataframe(frame, max_len) for frame in frames], axis=1)
                merged.to_excel(writer, sheet_name=safe_sheet_name(sheet, used_sheet_names), index=False)
        log_df = pd.DataFrame(log_rows)
        log_df.to_excel(writer, sheet_name="analysis_log", index=False)
    print("Excel file saved to:", excel_path)
    return results_store, excel_path, log_df


### Cell 07 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.

- Function descriptions: 
  - `condition_label(condition)`: Takes a condition name and returns a short label suitable for titles or legends.
  - `plot_results(results_store, conditions)`: Takes DRT-GPHT results for one sample and plots a four-column comparison figure: Nyquist, real-part Bode, imaginary-part Bode, and residuals.
  - `save_figure_index(figure_paths, excel_path)`: Summarizes sample names, condition names, and file paths for all saved figures and returns a figure-index table.


In [ ]:
def condition_label(condition: Condition) -> str:
    """generatefigurein of condition name."""
    if condition.kind == "raw":
        return "Raw Data"
    if condition.kind == "sparse":
        return f"Sparse (1/{condition.value})"
    if condition.kind == "noisy":
        return f"Noisy ({float(condition.value) * 100:g}%)"
    if condition.kind == "limited_range":
        low, high = condition.value
        return f"Limited ({low:g}-{high:g} Hz)"
    if condition.kind == "limited_percent":
        return f"Limited ({float(condition.value) * 100:g}%)"
    return condition.name
def plot_results(results_store: dict, conditions: list[Condition]) -> list[Path]:
    """plot and savesample of condition DRT-GPHT figure.
    Inputs:
        results_store: run_gpht_and_save() output of dictionary.
        conditions: conditionlist.
    Outputs:
        figure_paths: save of PNG pathlist.
    Purpose:
        plot Nyquist, Real Bode, Imaginary Bode threecolumnfigure; oneforone
        degradedcondition, and saverun index.
    """
    ensure_dir(FIG_DIR)
    figure_paths: list[Path] = []
    condition_map = {condition.name: condition for condition in conditions}
    for sheet, res_group in results_store.items():
        available_conditions = [condition for condition in conditions if condition.name in res_group]
        if not available_conditions:
            continue
        n_rows = len(available_conditions)
        fig, axes = plt.subplots(n_rows, 3, figsize=(34, max(9, 8 * n_rows)), constrained_layout=True)
        if n_rows == 1:
            axes = np.array([axes])
        raw_exp = res_group.get("raw", {}).get("exp", None)
        for row_idx, condition in enumerate(available_conditions):
            dtype = condition.name
            data = res_group[dtype]
            exp = data["exp"]
            gp = data["gp"]
            label = condition_label(condition_map[dtype])
            ax = axes[row_idx, 0]
            if raw_exp is not None:
                ax.plot(raw_exp["re"], -raw_exp["im"], color="black", alpha=0.18, lw=RAW_LINE_WIDTH, label="Raw reference")
            ax.plot(exp["re"], -exp["im"], "o", color="#D55E00", markersize=INPUT_MARKER_SIZE, alpha=0.75, label="Input data")
            ax.plot(gp["re"], -gp["im"], color="#0072B2", linewidth=GP_LINE_WIDTH, label="GP-HT mean")
            ax.set_ylabel(f"{label}\n-Z'' (Ohm)", fontfamily="Arial", fontsize=48, fontweight="bold")
            if row_idx == 0:
                ax.set_title("Nyquist Plot", fontfamily="Arial", fontsize=48)
            ax.legend(loc="best")
            ax.axis("equal")
            ax = axes[row_idx, 1]
            if raw_exp is not None:
                ax.semilogx(raw_exp["f"], raw_exp["re"], color="black", alpha=0.18, lw=RAW_LINE_WIDTH)
            ax.fill_between(gp["f"], gp["re"] - 3 * gp["sigma_re"], gp["re"] + 3 * gp["sigma_re"], facecolor="lightgrey", alpha=CI_ALPHA_3SIGMA, label=r"$\pm 3\sigma$")
            ax.fill_between(gp["f"], gp["re"] - 2 * gp["sigma_re"], gp["re"] + 2 * gp["sigma_re"], facecolor="silver", alpha=CI_ALPHA_2SIGMA, label=r"$\pm 2\sigma$")
            ax.fill_between(gp["f"], gp["re"] - gp["sigma_re"], gp["re"] + gp["sigma_re"], facecolor="grey", alpha=CI_ALPHA_1SIGMA, label=r"$\pm 1\sigma$")
            ax.semilogx(exp["f"], exp["re"], "o", color="#D55E00", markersize=INPUT_MARKER_SIZE, alpha=0.75)
            ax.semilogx(gp["f"], gp["re"], color="#0072B2", linewidth=GP_LINE_WIDTH)
            ax.set_ylabel("Z' (Ohm)", fontfamily="Arial", fontsize=48)
            if row_idx == 0:
                ax.set_title("Real Part Bode", fontfamily="Arial", fontsize=48)
                ax.legend(loc="best")
            ax = axes[row_idx, 2]
            if raw_exp is not None:
                ax.semilogx(raw_exp["f"], -raw_exp["im"], color="black", alpha=0.18, lw=RAW_LINE_WIDTH)
            ax.fill_between(gp["f"], -gp["im"] - 3 * gp["sigma_im"], -gp["im"] + 3 * gp["sigma_im"], facecolor="lightgrey", alpha=CI_ALPHA_3SIGMA)
            ax.fill_between(gp["f"], -gp["im"] - 2 * gp["sigma_im"], -gp["im"] + 2 * gp["sigma_im"], facecolor="silver", alpha=CI_ALPHA_2SIGMA)
            ax.fill_between(gp["f"], -gp["im"] - gp["sigma_im"], -gp["im"] + gp["sigma_im"], facecolor="grey", alpha=CI_ALPHA_1SIGMA)
            ax.semilogx(exp["f"], -exp["im"], "o", color="#D55E00", markersize=INPUT_MARKER_SIZE, alpha=0.75)
            ax.semilogx(gp["f"], -gp["im"], color="#0072B2", linewidth=GP_LINE_WIDTH)
            ax.set_ylabel("-Z'' (Ohm)", fontfamily="Arial", fontsize=48)
            if row_idx == 0:
                ax.set_title("Imaginary Part Bode", fontfamily="Arial", fontsize=48)
            if row_idx == n_rows - 1:
                axes[row_idx, 1].set_xlabel("Frequency (Hz)", fontfamily="Arial", fontsize=48)
                axes[row_idx, 2].set_xlabel("Frequency (Hz)", fontfamily="Arial", fontsize=48)
        for ax in axes.ravel():
            ax.tick_params(axis="both", labelsize=36, width=3, length=9)
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontfamily("Arial")
            for spine in ax.spines.values():
                spine.set_linewidth(3)
        figure_path = FIG_DIR / f"{safe_name(sheet)}_DRT_GPHT_imInput.png"
        fig.savefig(figure_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        figure_paths.append(figure_path)
        print("Figure saved to:", figure_path)
    return figure_paths
def save_figure_index(figure_paths: list[Path], excel_path: Path) -> None:
    """Save an index of generated figures.
    
    Inputs:
        figure_paths: list of saved figure paths.
    Outputs:
        DataFrame or CSV file listing figure paths.
    Purpose:
        Make batch-generated figures easier to locate and verify.
    """
    index_path = DATA_DIR / "figure_index.xlsx"
    pd.DataFrame({"figure_png": [str(path) for path in figure_paths]}).to_excel(index_path, index=False)
    print("Figure index saved to:", index_path)


### Cell 08 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: DRT-GPHT processing of experimental impedance data, comparison of four condition types, credible intervals, and result export.
- Main parameters: RUN_FULL_ANALYSIS.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.

In [ ]:
# Full-run switch: set RUN_FULL_ANALYSIS to False if you only want to inspect the configuration and functions.
RUN_FULL_ANALYSIS = True
if RUN_FULL_ANALYSIS:
    results_store, excel_path, log_df = run_gpht_and_save()
    figure_paths = plot_results(results_store, conditions)
    save_figure_index(figure_paths, excel_path)
    run_info = build_run_info()
    display(run_info)
    display(log_df.head())
    # Display one sample figure to quickly check plotting in the notebook.
    if figure_paths:
        print("Example figure:", figure_paths[0])
        display(Image(filename=str(figure_paths[0])))
else:
    print("Full analysis was skipped. Set RUN_FULL_ANALYSIS=True to generate Excel and PNG outputs.")
